# Smart School Project — Prédiction des échecs étudiants
Exploration des données (EDA) et comparaison de modèles de régression pour prédire le `score_examen`.

## Imports

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.sparse import issparse
from scipy.stats import gaussian_kde
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor

## Constantes et paramètres

In [ ]:
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860', '#DA8BC3']
MI_SAMPLE_SIZE = 50_000

small_dataset = False  # Mettre True pour tester sur 10 000 lignes seulement

## 1. Chargement des données

In [ ]:
if small_dataset:
    df = pd.read_csv('student_dataset/student_failure/train.csv', nrows=10000)
else:
    df = pd.read_csv('student_dataset/student_failure/train.csv')

df.head(10)

## 2. Préparation des données
`X_train_orig` conserve le DataFrame brut avant encodage (utilisé pour le feature engineering MI au graphe 5).

In [ ]:
y = df['score_examen']
X = df.drop(['score_examen', 'id', 'taille_etudiant'], axis=1)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, train_size=0.8, test_size=0.2, random_state=0
)
print('Shape avant preprocessing :', X_train.shape, X_valid.shape)

X_train_orig = X_train.copy()

## 3. Encodage des variables catégorielles
Changer `method` pour comparer les stratégies d'encodage.

| method | MAE full dataset |
|--------|------------------|
| `drop_categorical` | ≈ 8.63 |
| `ordinal_encoding_random` | ≈ 7.21 |
| `ordinal_encoding_smart` | ≈ 7.21 |
| `one_hot_encoding_1` | ≈ 7.21 |
| `one_hot_encoding_2` | ≈ 7.21 |
| `ordinal_and_one_hot_encoding` | ≈ 7.21 |

In [ ]:
method = 'ordinal_and_one_hot_encoding'

if method == 'drop_categorical':
    X_train = X_train.select_dtypes(exclude=['str'])
    X_valid = X_valid.select_dtypes(exclude=['str'])

elif method == 'ordinal_encoding_random':
    obj_mask = X_train.dtypes == 'str'
    object_cols = list(obj_mask[obj_mask].index)
    ordinal_encoder = OrdinalEncoder()
    X_train[object_cols] = ordinal_encoder.fit_transform(X_train[object_cols])
    X_valid[object_cols] = ordinal_encoder.transform(X_valid[object_cols])

elif method == 'ordinal_encoding_smart':
    manual_encoder = OrdinalEncoder(categories=[
        ['poor', 'average', 'good'],
        ['low',  'medium',  'high'],
        ['easy', 'moderate', 'hard']
    ])
    auto_encoder = OrdinalEncoder(categories='auto')
    preprocessor = ColumnTransformer(transformers=[
        ('ord_manuel', manual_encoder,
         ['qualité_sommeil', 'évaluation_établissement', 'difficulté_examen']),
        ('ord_auto',   auto_encoder,
         ['genre', 'diplôme', 'accès_internet', 'méthode_etude'])
    ], remainder='passthrough')
    X_train = preprocessor.fit_transform(X_train)
    X_valid = preprocessor.transform(X_valid)

elif method == 'one_hot_encoding_1':
    X_train = pd.get_dummies(X_train)
    X_valid = pd.get_dummies(X_valid)
    X_train, X_valid = X_train.align(X_valid, join='left', axis=1)

elif method == 'one_hot_encoding_2':
    obj_mask = X_train.dtypes == 'str'
    object_cols = list(obj_mask[obj_mask].index)
    one_hot_encoder = OneHotEncoder()
    preprocessor = ColumnTransformer(
        transformers=[('onehot', one_hot_encoder, object_cols)],
        remainder='passthrough'
    )
    X_train = preprocessor.fit_transform(X_train)
    X_valid = preprocessor.transform(X_valid)

elif method == 'ordinal_and_one_hot_encoding':
    manual_ordinal_cols = ['qualité_sommeil', 'évaluation_établissement', 'difficulté_examen']
    one_hot_cols        = ['genre', 'diplôme', 'accès_internet', 'méthode_etude']
    ordinal_encoder = OrdinalEncoder(categories=[
        ['poor', 'average', 'good'],
        ['low',  'medium',  'high'],
        ['easy', 'moderate', 'hard']
    ])
    one_hot_encoder = OneHotEncoder(sparse_output=False)
    preprocessor = ColumnTransformer(transformers=[
        ('ordinal', ordinal_encoder, manual_ordinal_cols),
        ('onehot',  one_hot_encoder, one_hot_cols)
    ], remainder='passthrough')
    X_train = preprocessor.fit_transform(X_train)
    X_valid = preprocessor.transform(X_valid)

else:
    raise ValueError("Méthode d'encodage inconnue.")

print('Shape après preprocessing :', X_train.shape, X_valid.shape)

In [ ]:
X_train_dense = X_train.toarray() if issparse(X_train) else np.array(X_train)
X_valid_dense = X_valid.toarray() if issparse(X_valid) else np.array(X_valid)

imputer = SimpleImputer(strategy='median')
X_train_dense = imputer.fit_transform(X_train_dense)
X_valid_dense = imputer.transform(X_valid_dense)

## 4. Entraînement des modèles

In [ ]:
baseline_pred = np.full(len(y_valid), y_train.mean())
mae_baseline  = mean_absolute_error(baseline_pred, y_valid)
print(f'Baseline (moyenne)         MAE : {mae_baseline:.4f}')

In [ ]:
lr = LinearRegression()
lr.fit(X_train_dense, y_train)
mae_lr = mean_absolute_error(lr.predict(X_valid_dense), y_valid)
print(f'Régression linéaire        MAE : {mae_lr:.4f}')

In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=4)
rf.fit(X_train_dense, y_train)
mae_rf = mean_absolute_error(rf.predict(X_valid_dense), y_valid)
print(f'Random Forest              MAE : {mae_rf:.4f}')

In [ ]:
model_xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    n_jobs=4,
    early_stopping_rounds=50,
    eval_metric='rmse'
)
model_xgb.fit(
    X_train_dense, y_train,
    eval_set=[(X_valid_dense, y_valid)],
    verbose=False
)
mae_xgb = mean_absolute_error(model_xgb.predict(X_valid_dense), y_valid)
print(f'XGBoost                    MAE : {mae_xgb:.4f}')

In [ ]:
scaler_mlp = StandardScaler()
X_train_scaled = scaler_mlp.fit_transform(X_train_dense)
X_valid_scaled = scaler_mlp.transform(X_valid_dense)

mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    max_iter=300,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20
)
mlp.fit(X_train_scaled, y_train)
mae_mlp = mean_absolute_error(mlp.predict(X_valid_scaled), y_valid)
print(f'MLP (réseau de neurones)   MAE : {mae_mlp:.4f}')

## 5. Graphes EDA
Initialisation commune : thème seaborn et noms de features partagés entre les graphes 4, 5 et 8.

In [ ]:
sns.set_theme(style='whitegrid')

try:
    raw_names = preprocessor.get_feature_names_out()
    feat_names_base = [n.split('__', 1)[-1] for n in raw_names]
except Exception:
    feat_names_base = [f'feature_{i}' for i in range(X_train_dense.shape[1])]

### Graphe 1 — Valeurs manquantes par colonne

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

fig0, ax0 = plt.subplots(figsize=(10, 5))
if missing.empty:
    ax0.text(0.5, 0.5, 'Aucune valeur manquante dans le dataset',
             ha='center', va='center', fontsize=14, color='green',
             transform=ax0.transAxes)
else:
    bars0 = ax0.bar(missing.index, missing.values, color=PALETTE[3], alpha=0.85)
    for bar, val in zip(bars0, missing.values):
        ax0.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + missing.max() * 0.01,
                 f'{val / len(df):.1%}',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax0.set_ylabel('Nombre de valeurs manquantes', fontsize=11)
    plt.xticks(rotation=30, ha='right')

ax0.set_title('Valeurs manquantes par colonne', fontsize=13, fontweight='bold', pad=12)
ax0.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

### Graphe 2 — Distribution des scores d'examen

In [ ]:
s = df['score_examen']
n_total = len(s)
n_100   = (s == 100).sum()
n_fail  = (s < 50).sum()
n_19    = (s == 19).sum()

counts, edges = np.histogram(s, bins=np.arange(12, 102, 1))

fig1, ax1 = plt.subplots(figsize=(14, 7))

bar_colors = []
for left, right in zip(edges[:-1], edges[1:]):
    mid = (left + right) / 2
    if mid == 19:    bar_colors.append('#E84393')
    elif mid == 100: bar_colors.append('#DD8452')
    elif mid < 50:   bar_colors.append('#E05C5C')
    else:            bar_colors.append('#4C72B0')

ax1.bar(edges[:-1], counts, width=0.9, color=bar_colors, alpha=0.85, align='edge', zorder=2)

s_kde = s[(s != 100) & (s != 19)]
kde = gaussian_kde(s_kde, bw_method=0.08)
x_kde = np.linspace(12, 99, 400)
kde_vals = kde(x_kde) * len(s_kde)
ax1.plot(x_kde, kde_vals, color='navy', linewidth=2.0, label='KDE (hors anomalies)', zorder=3)

ax1.axvline(50, color='red', linestyle='--', linewidth=2,
            label=f'Seuil échec 50 — {n_fail:,} étudiants ({n_fail/n_total:.1%})', zorder=4)

idx_19 = np.where(edges[:-1] == 19)[0]
if len(idx_19):
    h19 = counts[idx_19[0]]
    ax1.annotate(
        f'Anomalie\nscore=19\nn={n_19:,}',
        xy=(19.45, h19), xytext=(24, h19 * 0.88),
        fontsize=9, color='#E84393', fontweight='bold',
        arrowprops=dict(arrowstyle='->', color='#E84393', lw=1.5),
        bbox=dict(boxstyle='round,pad=0.3', fc='#fff0f7', ec='#E84393', lw=1)
    )

ax1.annotate(
    f'Plafonnement\nscore=100\nn={n_100:,} ({n_100/n_total:.1%})',
    xy=(100.45, n_100), xytext=(90, n_100 * 0.92),
    fontsize=9, color='#DD8452', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#DD8452', lw=1.5),
    bbox=dict(boxstyle='round,pad=0.3', fc='#fff8f0', ec='#DD8452', lw=1)
)

legend_elements = [
    mpatches.Patch(facecolor='#4C72B0', alpha=0.85, label='Zone réussite (≥50)'),
    mpatches.Patch(facecolor='#E05C5C', alpha=0.85, label=f'Zone échec (<50) — {n_fail/n_total:.1%}'),
    mpatches.Patch(facecolor='#E84393', alpha=0.85, label=f'Anomalie score=19 — n={n_19:,}'),
    mpatches.Patch(facecolor='#DD8452', alpha=0.85, label=f'Plafonnement score=100 — {n_100/n_total:.1%}'),
    plt.Line2D([0], [0], color='navy', linewidth=2, label='KDE (hors anomalies)'),
    plt.Line2D([0], [0], color='red', linestyle='--', linewidth=2, label='Seuil échec (50)'),
]
ax1.legend(handles=legend_elements, loc='upper left', fontsize=9, framealpha=0.9)
ax1.set_title(
    f"Distribution des scores d'examen  —  n={n_total:,}"
    f'  |  moyenne={s.mean():.1f}  |  médiane={s.median():.0f}',
    fontsize=14, fontweight='bold', pad=14
)
ax1.set_xlabel("Score d'examen", fontsize=12)
ax1.set_ylabel("Nombre d'étudiants", fontsize=12)
ax1.set_xlim(10, 102)
ax1.set_xticks(range(10, 101, 5))
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax1.grid(axis='y', alpha=0.4, zorder=0)
plt.tight_layout()
plt.show()

### Graphe 3 — Score d'examen selon les variables catégorielles

In [ ]:
cat_cols = [
    'diplôme', 'méthode_etude', 'accès_internet',
    'qualité_sommeil', 'difficulté_examen', 'évaluation_établissement'
]
fig3, axes3 = plt.subplots(2, 3, figsize=(16, 10))

for ax, col in zip(axes3.flat, cat_cols):
    order = df.groupby(col)['score_examen'].median().sort_values().index
    sns.boxplot(data=df, x=col, y='score_examen',
                order=order, ax=ax, hue=col, palette='Set2', legend=False)
    ax.axhline(50, color='red', linestyle='--', linewidth=1.2, label='Seuil échec (50)')
    ax.set_title(f'Score selon {col}', fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel("Score d'examen", fontsize=9)
    ax.tick_params(axis='x', rotation=15)
    ax.legend(fontsize=8)

fig3.suptitle(
    "Distribution du score d'examen selon les variables catégorielles",
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

### Graphe 4 — Matrice de corrélation (toutes les variables après encodage)
Réutilise `X_train_dense` et `feat_names_base` — aucun ré-encodage nécessaire. `score_examen` est ajouté comme dernière colonne.

In [ ]:
df_corr = pd.DataFrame(X_train_dense, columns=feat_names_base)
df_corr['score_examen'] = y_train.values

corr_matrix = df_corr.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

n_feats  = len(df_corr.columns)
fig_w    = max(12, n_feats * 0.65)
fig_h    = max(10, n_feats * 0.55)
annot_sz = max(5, 9 - n_feats // 6)

fig4, ax4 = plt.subplots(figsize=(fig_w, fig_h))
sns.heatmap(
    corr_matrix, mask=mask,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.3, ax=ax4,
    annot_kws={'size': annot_sz}
)
ax4.set_title(
    'Matrice de corrélation (Pearson) — toutes les variables (après encodage)',
    fontsize=13, fontweight='bold', pad=12
)
ax4.tick_params(axis='x', rotation=45, labelsize=max(6, annot_sz))
ax4.tick_params(axis='y', labelsize=max(6, annot_sz))
plt.tight_layout()
plt.show()

### Graphe 5 — Importance des features (Mutual Information)
Réutilise `X_train_dense` + 3 features engineered calculées depuis `X_train_orig`.

In [ ]:
print('Feature engineering pour Mutual Information...')

he = X_train_orig['heures_etude'].fillna(X_train_orig['heures_etude'].median())
ratio_ef   = (he / (X_train_orig['heures_fête'] + 1)).values.reshape(-1, 1)
bien_etre  = (X_train_orig['heures_sommeil'] / 12.0).values.reshape(-1, 1)
engagement = (X_train_orig['assiduité_classe'] * he).values.reshape(-1, 1)

X_mi_arr = np.hstack([X_train_dense, ratio_ef, bien_etre, engagement])
mi_feature_names = feat_names_base + ['ratio_etude_fete', 'score_bien_etre', 'engagement']

X_mi = pd.DataFrame(X_mi_arr, columns=mi_feature_names).fillna(0).astype(float)
y_mi = (y_train < 50).astype(int).reset_index(drop=True)

n_rows = len(X_mi)
if n_rows > MI_SAMPLE_SIZE:
    print(f'Échantillonnage : {MI_SAMPLE_SIZE:,} / {n_rows:,} lignes '
          f'({MI_SAMPLE_SIZE/n_rows:.0%}) pour accélérer le calcul MI...')
    sample_idx = (
        pd.DataFrame({'y': y_mi})
        .groupby('y', group_keys=False)
        .apply(lambda g: g.sample(
            n=int(MI_SAMPLE_SIZE * len(g) / n_rows),
            random_state=42
        ), include_groups=False)
        .index
    )
    X_mi_sample = X_mi.loc[sample_idx]
    y_mi_sample = y_mi.loc[sample_idx]
else:
    X_mi_sample = X_mi
    y_mi_sample = y_mi

print(f'Calcul Mutual Information ({len(mi_feature_names)} features, {len(X_mi_sample):,} lignes)...')
mi = mutual_info_classif(X_mi_sample, y_mi_sample, random_state=42, n_neighbors=3)
mi_series = pd.Series(mi, index=mi_feature_names).sort_values(ascending=True)
print('Calcul MI terminé.')

fig5, ax5 = plt.subplots(figsize=(10, 9))
mi_colors = [PALETTE[3] if v > mi_series.median() else PALETTE[0] for v in mi_series.values]
ax5.barh(mi_series.index, mi_series.values, color=mi_colors)
ax5.axvline(mi_series.median(), color='red', linestyle='--', linewidth=1.5)
legend_mi = [
    mpatches.Patch(facecolor=PALETTE[3], label='Score > médiane'),
    mpatches.Patch(facecolor=PALETTE[0], label='Score ≤ médiane'),
    plt.Line2D([0], [0], color='red', linestyle='--', linewidth=1.5, label='Médiane'),
]
ax5.legend(handles=legend_mi, fontsize=9, framealpha=0.9)
ax5.set_title('Importance des features (Mutual Information)', fontsize=13, fontweight='bold', pad=12)
ax5.set_xlabel('Score MI', fontsize=11)
ax5.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.show()

## 6. Résultats des modèles

### Graphe 6 — Comparaison des modèles (MAE)

In [ ]:
model_names = [
    'Baseline\n(moyenne)',
    'Régression\nLinéaire',
    'Random\nForest',
    'XGBoost',
    'MLP\n(neurones)'
]
model_maes = [mae_baseline, mae_lr, mae_rf, mae_xgb, mae_mlp]

best_idx  = int(np.argmin(model_maes))
bar_cols6 = [PALETTE[1] if i == best_idx else PALETTE[0] for i in range(len(model_maes))]

fig6, ax6 = plt.subplots(figsize=(11, 6))
bars6 = ax6.bar(model_names, model_maes, color=bar_cols6, alpha=0.85, width=0.5)

for bar, mae in zip(bars6, model_maes):
    ax6.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(model_maes) * 0.01,
        f'{mae:.3f}',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax6.axhline(mae_baseline, color='red', linestyle='--', linewidth=1.5,
            label=f'Baseline MAE = {mae_baseline:.3f}')
ax6.set_title(
    'Comparaison des modèles — Mean Absolute Error (MAE)\n(plus bas = meilleur)',
    fontsize=13, fontweight='bold', pad=12
)
ax6.set_ylabel('MAE', fontsize=11)
ax6.legend(fontsize=10)
ax6.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

### Graphe 7 — Courbe d'apprentissage XGBoost (RMSE vs rounds)

In [ ]:
results = model_xgb.evals_result()
epochs  = len(results['validation_0']['rmse'])

fig7, ax7 = plt.subplots(figsize=(10, 5))
ax7.plot(range(epochs), results['validation_0']['rmse'],
         color=PALETTE[0], linewidth=2, label='Validation RMSE')

best_round = int(np.argmin(results['validation_0']['rmse']))
best_rmse  = results['validation_0']['rmse'][best_round]
ax7.axvline(best_round, color='red', linestyle='--', linewidth=1.5,
            label=f'Meilleur round : {best_round}  (RMSE={best_rmse:.4f})')
ax7.scatter([best_round], [best_rmse], color='red', zorder=5, s=60)
ax7.set_xlabel('Nombre de rounds de boosting', fontsize=11)
ax7.set_ylabel('RMSE', fontsize=11)
ax7.set_title("Courbe d'apprentissage XGBoost — RMSE sur validation",
              fontsize=13, fontweight='bold', pad=12)
ax7.legend(fontsize=10)
ax7.grid(alpha=0.4)
plt.tight_layout()
plt.show()

### Graphe 8 — Importance des features XGBoost (gain) — Top 20

In [ ]:
fi_vals   = model_xgb.feature_importances_
fi_series = pd.Series(fi_vals, index=feat_names_base).sort_values(ascending=True)
fi_top    = fi_series.tail(20)

fig8, ax8 = plt.subplots(figsize=(10, 8))
fi_colors = [PALETTE[3] if v > fi_top.median() else PALETTE[0] for v in fi_top.values]
ax8.barh(fi_top.index, fi_top.values, color=fi_colors)
ax8.axvline(fi_top.median(), color='red', linestyle='--', linewidth=1.5, label='Médiane')
legend_fi = [
    mpatches.Patch(facecolor=PALETTE[3], label='Importance > médiane'),
    mpatches.Patch(facecolor=PALETTE[0], label='Importance ≤ médiane'),
    plt.Line2D([0], [0], color='red', linestyle='--', linewidth=1.5, label='Médiane'),
]
ax8.legend(handles=legend_fi, fontsize=9, framealpha=0.9)
ax8.set_title('Importance des features XGBoost (gain) — Top 20',
              fontsize=13, fontweight='bold', pad=12)
ax8.set_xlabel('Importance (gain moyen par split)', fontsize=11)
ax8.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.show()